In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)  # Toon alle kolommen

In [2]:

#df_=pd.read_excel(r'C:\Users\HMUSS\OneDrive - Albeda\Bureaublad\Analyse 2024-2025_deel 2\Eda\Results.xlsx')
dfcsv=pd.read_csv(r'C:\Users\HMUSS\OneDrive - Albeda\Bureaublad\Npulse\Dataset28052026.csv')
crebo=pd.read_excel(r'C:\Users\HMUSS\OneDrive - Albeda\Bureaublad\Npulse\crebokoppeltabel.xlsx')

dfcsv=dfcsv.sort_values(['Deelnemernummer','Studiejaar','niveau'])


In [3]:

def verwerk_student_data(df_, crebo):
    # CREBO info koppelen
    crebo = crebo.rename(columns={
        'opleidingscode (crebo)': 'crebo',
        'beroepsopleiding_id': "BC_code",
        'beroepsopleiding (dossier)': 'dossier'
    })[['crebo', 'BC_code', 'dossier']]
    df = df_.merge(crebo, on='crebo', how='left')

    # Filter op geldige data
    df = df[(df.Cluster != 'Cluster Techniek') & (df.crebo != 99999)].copy()

    # Relevante kolommen selecteren (zonder geboortedatum)
    kolommen = [
        'Studiejaar', 'begindatum', 'einddatum', 'Deelnemernummer','Volgnummer',
        'Deelnemer_geslacht', 'Leerweg', 'crebo', 'niveau', 'BC_code', 'Team',
        'College', 'Cluster', 'geslaagd', 'dossier', 'intensiteit', 'locatie', 'afkorting'
    ]
    df = df[kolommen].drop_duplicates()
    # Geslaagd per student per niveau en BC_code
    diploma = df.groupby(['Deelnemernummer'])['geslaagd'].max().reset_index(name='geslaagd_per_student')
    df = df.merge(diploma, on=['Deelnemernummer'], how='left')

    # Geslaagd per niveau (alle BC_codes)
    max_diplomas = df.groupby(['Deelnemernummer'])['geslaagd'].max().reset_index()
    totaal_diplomas = max_diplomas.groupby('Deelnemernummer')['geslaagd'].sum().reset_index(name='totaal_diplomas')
    geslaagd_niveau = max_diplomas.rename(columns={'geslaagd': 'geslaagd_niveau'})

    df = df.merge(totaal_diplomas, on='Deelnemernummer', how='left')
    df = df.merge(geslaagd_niveau, on=['Deelnemernummer'], how='left')
    # Voorbereiding op telling per 1 oktober
    df['Studiejaar'] = df['Studiejaar'].str[:4].astype(float)
    df = df.sort_values(['Deelnemernummer', 'Studiejaar'])
    df['1_oktober'] = pd.to_datetime(df['Studiejaar'].astype(int).astype(str) + '-10-01')
    df['telling'] = ((df['1_oktober'] >= df['begindatum']) & 
                     ((df['1_oktober'] < df['einddatum']) | df['einddatum'].isna())).astype(int)
    df = df[df['telling'] == 1]
    df=df.sort_values(['Deelnemernummer','Studiejaar'])
    df=df.drop_duplicates(subset=['Studiejaar','Deelnemernummer'], keep='last')
    df=df.sort_values(['Deelnemernummer','Studiejaar'])
    # Jaarlijkse doorstroom en instroombepaling
    df['vorig_studiejaar'] = df.groupby('Deelnemernummer')['Studiejaar'].shift(1)
    df['studiejaar_verschil'] = df['Studiejaar'] - df['vorig_studiejaar']
    df['instroom'] = ((df['studiejaar_verschil'] > 1) | df['vorig_studiejaar'].isna()).astype(int)
    df['instroom_groep'] = df.groupby('Deelnemernummer')['instroom'].cumsum()
    
    df['jaar_nummer'] = df.groupby(['Deelnemernummer', 'instroom_groep']).cumcount() + 1
    ##doorstroom
    df['doorstroom'] = np.where(df['studiejaar_verschil'] == 1, 1, 0)

    # Uitstroom met of zonder diploma
    df['uitstroom'] = df['studiejaar_verschil'].shift(-1) > 1
    df['laatste_inschrijving'] = df.groupby('Deelnemernummer')['Studiejaar'].transform('max') == df['Studiejaar']
    df['uitstroom'] = np.where(df['uitstroom'] | df['laatste_inschrijving'], 1, 0)
    df['uitstroom_telling'] = df.groupby(['Studiejaar', 'Deelnemernummer', 'uitstroom']).cumcount() + 1
    
    # Duur berekening
    df['begindatum'] = pd.to_datetime(df['begindatum'])
    df['einddatum'] = pd.to_datetime(df['einddatum'])
    df['duur_deelname'] = (df['einddatum'] - df['begindatum']).dt.days / 364.25

        # Nieuwe kolom: studieduur in studiejaren per instroomgroep (van instroom tot uitstroom)
    df['studieduur_studiejaren'] = df.groupby(['Deelnemernummer', 'instroom_groep'])['Studiejaar'].transform('max') - \
                                    df.groupby(['Deelnemernummer', 'instroom_groep'])['Studiejaar'].transform('min') + 1
    # Maak de nieuwe kolom 'uitstroom_'
    df['uitstroom_'] = np.where(df.einddatum.isnull(), 0,    np.where( (df['uitstroom'] == 1) & (df['uitstroom_telling'] == 1), 1, 0   ))
    #df['uitstroom_met_diploma'] = ((df['geslaagd_per_student'] > 0 ) & (df['uitstroom_'] == 1)).astype(int)
   # df['uitstroom_zonder_diploma'] = (((df['geslaagd_per_student'] == 0) ) & (df['uitstroom_'] == 1)).astype(int)
    #dubbele regels verwijderen
    #df=df[df.telling ==1].groupby(['Deelnemernummer','Volgnummer','Studiejaar','niveau','telling','instroom_groep'])[['instroom','doorstroom','uitstroom_','uitstroom_zonder_diploma','uitstroom_met_diploma','geslaagd_per_student']].max().reset_index()
    df=df[df.telling ==1].groupby(['Deelnemernummer','Volgnummer','Studiejaar','niveau','telling','instroom_groep'])[['instroom','doorstroom','uitstroom_']].max().reset_index()
    df=df.sort_values(['Deelnemernummer','Studiejaar'])
    df['jaar_nummer'] = df.groupby(['Deelnemernummer', 'instroom_groep']).cumcount() + 1
    #tussenjaren te kunnen berekenen
    #df['begin_tussenjaar'] = df.groupby('Deelnemernummer')['instroom_groep'].shift(-1)
   # df['volgend_jaar_nummer'] = df.groupby('Deelnemernummer')['jaar_nummer'].shift(-1)
   # df['eind_tussenjaar'] = df.groupby('Deelnemernummer')['instroom_groep'].shift(1)
   # df['tussenjaar']=np.where((df['begin_tussenjaar'] > df['instroom_groep']) &  (df['instroom']==1),1,0)
   # df['herstart_na_tussenjaar']=np.where((df['eind_tussenjaar'] < df['instroom_groep']) &(df['instroom']==1),1,0)

    return df



In [4]:
# Stap 1: initieel verwerken
df_clean = verwerk_student_data(dfcsv, crebo)

In [5]:
df_clean

,Deelnemernummer,Volgnummer,Studiejaar,niveau,telling,instroom_groep,instroom,doorstroom,uitstroom_,jaar_nummer
0,10155031,3,2013.0,Niveau2,1,1,1,0,1,1
1,10166028,8,2013.0,Niveau4,1,1,1,0,1,1
2,10179042,2,2022.0,Niveau3,1,1,1,0,0,1
3,10179042,2,2023.0,Niveau3,1,1,0,1,0,2
4,10179042,4,2024.0,Niveau4,1,1,0,1,0,3
...,...,...,...,...,...,...,...,...,...,...
236417,400210081,1,2025.0,Niveau4,1,1,1,0,1,1
236418,400210082,1,2025.0,Niveau1,1,1,1,0,0,1
236419,400210107,1,2025.0,Niveau4,1,1,1,0,1,1
236420,400210112,1,2025.0,Niveau2,1,1,1,0,0,1


In [6]:
df_clean.groupby(['Studiejaar'])[['instroom','doorstroom','uitstroom_']].sum().reset_index()

,Studiejaar,instroom,doorstroom,uitstroom_
0,2013.0,18385,0,7689
1,2014.0,7180,10696,7305
2,2015.0,6946,10571,7184
3,2016.0,6889,10333,6939
4,2017.0,7206,10283,6981
5,2018.0,7404,10508,7048
6,2019.0,7518,10864,6885
7,2020.0,7799,11497,7406
8,2021.0,7106,11890,7912
9,2022.0,7431,11084,7715


In [7]:
import numpy as np

def genereer_markov_statusen(df):

    # Sorteren en uniek maken per deelnemer per studiejaar
    df = df.sort_values(['Deelnemernummer', 'Studiejaar'])
    df = df.drop_duplicates(
        subset=['Deelnemernummer', 'Studiejaar'],
        keep='last'
    )

    # Nieuw traject bepalen bij niveauwissel of instroom
    df['niveau_shift'] = df.groupby('Deelnemernummer')['niveau'].shift()

    df['nieuw_traject'] = (  (df['niveau'] != df['niveau_shift']) |(df['instroom'] == 1))

    df['traject_id'] = ( df.groupby('Deelnemernummer')['nieuw_traject'] .cumsum())

    # Jaar binnen traject
    df['jaar_nummer'] = (   df.groupby(['Deelnemernummer', 'traject_id'])   .cumcount() + 1)

    # Volgende info
    df['jaar_nummer_volgend'] = (df.groupby('Deelnemernummer')['jaar_nummer'].shift(-1))

    df['niveau_volgend'] = ( df.groupby('Deelnemernummer')['niveau'] .shift(-1) )

    # =========================
    # HUIDIGE STATUS (geen uitstroom!)
    # =========================
    df['huidige_status'] = np.select(
        [
            (df['jaar_nummer'] == 1),
            (df['jaar_nummer'] == 2),
            (df['jaar_nummer'] == 3),
            (df['jaar_nummer'] == 4),
            (df['jaar_nummer'] == 5),
            (df['jaar_nummer'] > 5),
        ],
        [
            'jaar_1',
            'jaar_2',
            'jaar_3',
            'jaar_4',
            'jaar_5',
            'jaar_>5',
        ],
        default='onbekend'
    )

    # =========================
    # VOLGENDE STATUS (hier zit uitstroom WEL)
    # =========================
    df['volgende_status'] = np.select(
        [
            df['uitstroom_'] == 1,  # <-- uitstroom is toekomststate
            df['niveau_volgend'].isna(),
            (df['niveau_volgend'] != df['niveau']),
            (df['jaar_nummer_volgend'] == 1),
            (df['jaar_nummer_volgend'] == 2),
            (df['jaar_nummer_volgend'] == 3),
            (df['jaar_nummer_volgend'] == 4),
            (df['jaar_nummer_volgend'] == 5),
            (df['jaar_nummer_volgend'] > 5),
        ],
        [
            'uitstroom',
            np.nan,
            'jaar_1',
            'jaar_1',
            'jaar_2',
            'jaar_3',
            'jaar_4',
            'jaar_5',
            'jaar_>5',
        ],
        default='onbekend'
    )

    # =========================
    # COMBINATIES (state space Markov)
    # =========================

    df['status_combi'] = np.where( df['huidige_status'] == 'uitstroom','uitstroom',  df['niveau'].astype(str) + '_' + df['huidige_status'])

    df['status_combi'] = np.where( df['instroom'] == 1,   df['status_combi'] + '_nieuw', df['status_combi'])

    df['volgende_status_combi'] = np.where(  df['volgende_status'].isna() | (df['volgende_status'] == 'uitstroom'), df['volgende_status'], df['niveau_volgend'].astype(str) + '_' + df['volgende_status'] )

    
    #statussen samen voegen bij niveau 1 en 2 >2 jaar en niveau 3 >3 is hoogste status

    #df['status_combi'] =np.where(((df['niveau'] == 'Niveau1') & (df['jaar_nummer']  >2)),'Niveau1_jaar_>2',df['status_combi']) 
    #df['status_combi'] =np.where(((df['niveau'] == 'Niveau2') & (df['jaar_nummer']  >2)),'Niveau1_jaar_>2',df['status_combi']) 
    #df['status_combi'] =np.where(((df['niveau'] == 'Niveau3') & (df['jaar_nummer']  >3)),'Niveau1_jaar_>3',df['status_combi']) 

    #df['volgende_status_combi'] =np.where(df['niveau_volgend'] == 'Niveau1' & df['jaar_nummer_volgend']  >2,'Niveau1_jaar_>2',df['volgende_status_combi']) 
    #df['volgende_status_combi'] =np.where(df['niveau_volgend'] == 'Niveau2' & df['jaar_nummer_volgend']  >2,'Niveau1_jaar_>2',df['volgende_status_combi']) 
   # df['volgende_status_combi'] =np.where(df['niveau_volgend'] == 'Niveau3' & df['jaar_nummer_volgend']  >3,'Niveau1_jaar_>3',df['volgende_status_combi']) 

    return df

In [8]:
df_markov = genereer_markov_statusen(df_clean)

# samenvoegen statussen
df_markov['status_combi'] =np.where(((df_markov['niveau'] == 'Niveau1') & (df_markov['jaar_nummer']  >2)),'Niveau1_jaar_>2',df_markov['status_combi']) 
df_markov['status_combi'] =np.where(((df_markov['niveau'] == 'Niveau2') & (df_markov['jaar_nummer']  >2)),'Niveau2_jaar_>2',df_markov['status_combi']) 
df_markov['status_combi'] =np.where(((df_markov['niveau'] == 'Niveau3') & (df_markov['jaar_nummer']  >3)),'Niveau3_jaar_>3',df_markov['status_combi']) 

df_markov['volgende_status_combi'] =np.where(((df_markov['niveau_volgend'] == 'Niveau1') & (df_markov['jaar_nummer_volgend']  >2)),'Niveau1_jaar_>2',df_markov['volgende_status_combi']) 
df_markov['volgende_status_combi'] =np.where(((df_markov['niveau_volgend'] == 'Niveau2') & (df_markov['jaar_nummer_volgend']  >2)),'Niveau2_jaar_>2',df_markov['volgende_status_combi']) 
df_markov['volgende_status_combi'] =np.where(((df_markov['niveau_volgend'] == 'Niveau3') & (df_markov['jaar_nummer_volgend']  >3)),'Niveau3_jaar_>3',df_markov['volgende_status_combi']) 
 

In [9]:
df_markov=df_markov[[ 'Studiejaar','Deelnemernummer','telling','niveau','instroom','doorstroom','uitstroom_','status_combi','volgende_status_combi']]

In [10]:
df_markov.to_csv(r'input_markov_chain_model.csv')